# Batchsimulation mit Python

## Voraussetzungen

Windows mit Microsoft Excel, eine gespeicherte PEExcel-Arbeitskopie und vorbereitete Szenarien.
Installiere im Repository `python -m pip install -e ".[batch]"`.
Die Arbeitsmappe benötigt `RunScenariosAutomated(reloadBase As Boolean) As Boolean`:
VBA lädt optional die Basis, wendet Presets an, berechnet und aktualisiert den Status.
VBA speichert und exportiert dabei nicht. Python erfasst IN/STAGE und OUT/Live sofort.
Für Stundenwerte wird zusätzlich die strukturierte Tabelle SIM gelesen.

Dieses Tutorial verwendet `Project` und die öffentliche API `pexl.batch.run_batch`.
Es benötigt keine direkten Excel-Automationsaufrufe. Die Berechnung ist zunächst ausgeschaltet;
zum Ausführen Pfade prüfen und `RUN_EXCEL = True` setzen. Vorher die Arbeitskopie speichern und in Excel schließen. Der Batch öffnet eine eigene Excel-Instanz.
Während des Laufs keine Eingaben in Excel ändern.

### Aufgabenteilung in pexl

`pexl.batch.run_batch` organisiert den Batch-Auftrag: Szenarienauswahl und Reihenfolge,
Ergebnisdateien, Fortschritt und Wiederaufnahme. `Project` und `Scenario` enthalten die Daten.

Der allgemeine `pexl.io.excel_com.PEExcelAdapter` übernimmt die Verbindung zur Arbeitsmappe
und den Zugriff auf Excel-Tabellen. Szenarien lesen, konfigurieren, ausführen und aktuelle
Ergebnisse erfassen sind seine bisher implementierten Operationen. Die Verbindung selbst
setzt keine SCEN-Tabelle voraus; Tabellen werden erst bei Bedarf angesprochen. So können
künftige PEExcel-Funktionen dieselbe Adapter-Infrastruktur verwenden.

Für dieses Tutorial genügt `run_batch`; ein direkter Adapter- oder COM-Aufruf ist nicht nötig.


In [ ]:
from pathlib import Path
import pandas as pd
from pexl import Project
from pexl.batch import run_batch, inspect_batch

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/pexl").is_dir())
workbook = ROOT / "excel/development/ka_PEQ_Nachweistool_DEV.xlsb"
output_dir = ROOT / "notebooks/local_exports/batch_api"
RUN_EXCEL = False
scenario_source = None  # None: vorhandene SCEN-Tabelle; alternativ Pfad zu CSV/XLSX
result = None

DEFAULT_BASE = "Default"  # exakter IN-Spaltenname in ka_PEQ_Nachweistool_DEV.xlsb
RUN_INSPECTION = False  # liest/validiert Excel; startet keine Simulation
RUN_RESUME = False
RUN_RETRY = False

## 1. Szenarien aus Excel oder einer externen Liste

Ohne `scenario_source` werden die Zeilen der vorhandenen Tabelle `Scenarios` auf SCEN verwendet.
Die API wählt anhand des Filters aus; die bisherige Spalte `Simulieren?` begrenzt den Lauf nicht.
Pro VBA-Aufruf wird genau ein Szenario ausgewählt.

Alternativ eine CSV/XLSX mit denselben SCEN-Spaltennamen angeben. Pflichtfelder sind
`Scenario Name` und `Base Scenario`, dazu kommen die gewünschten Preset-Spalten mit ihren
exakten Namen aus der Arbeitsmappe. Unbekannte Spalten werden abgewiesen. Jede Zeile braucht
einen eindeutigen Szenarionamen. Die API schreibt die Liste temporär nach SCEN und stellt
anschließend die ursprüngliche Konfiguration einschließlich Formeln wieder her.

Eine Excel-Liste verwendet das Blatt `Scenarios` oder `SCEN`, alternativ genau ein Blatt.
CSV-Dateien besitzen eine Kopfzeile. Beispielkonfiguration (Pfad an eigene Liste anpassen):

```python
scenario_source = ROOT / "data/archetypes.csv"
# oder: scenario_source = ROOT / "data/archetypes.xlsx"
# oder direkt: scenario_source = pd.DataFrame(...)
```

Für den ersten Live-Test nur 2–5 Szenarien verwenden. Ein eigener Ausgabeordner je
Workbook-/Konfigurationsstand verhindert die Vermischung unterschiedlicher Modellstände.

Auswahl, Save-/Exportflags und `Skip base reload?` werden durch die API gesteuert;
die entsprechenden Listeneinträge werden dafür vorübergehend überschrieben.
`Zeitpunkt` und Statusfelder gehören nicht zur Resume-Eingabeidentität.

## Gebäudetypologien auf Basis von `Default` vorbereiten

Die vorhandene Entwicklungsmappe `excel/development/ka_PEQ_Nachweistool_DEV.xlsb` besitzt die Basis **`Default`** in IN. Nicht `default`, `DEFAULT` oder den Anzeigenamen eines Ergebnisses verwenden. Vorhandene Typologiezeilen auf SCEN kombinieren diese Basis mit `Presets_geometry_table`, `Presets_hulls_table`, Heizsystem, PV und weiteren Presets. Ihre Namen enthalten beispielsweise `EFH_1G|1945-59|WP_Sole_fc|PV30|noFlex`.

`inspect_batch()` verwendet dieselben Lade-, Auswahl- und Validierungsfunktionen wie `run_batch()`. Es prüft Basis- und Presetreferenzen gegen die gespeicherte Arbeitsmappe. Die folgende Auswahl übernimmt vollständige SCEN-Zeilen einschließlich aller `Custom *`-Spalten. Keine Rekonstruktion aus Ergebnisnamen und kein eigenes Typologieschema.

**Wichtig:** `excel/development/Szenarien_5400.csv` ist eine vorgelagerte Typologieliste (`Szenario_ID`, `Quartier`, …), keine direkt verwendbare SCEN-Datei. Für diesen Ablauf die bereits vorbereiteten SCEN-Zeilen der Entwicklungsmappe oder einen Export mit den originalen SCEN-Spalten verwenden.

Zunächst `RUN_INSPECTION = True` setzen und die Auswahl ansehen. `RUN_EXCEL` bleibt dabei False. Für neue Varianten eine bestehende vollständige Zeile kopieren, einen eindeutigen `Scenario Name` vergeben, `Base Scenario = "Default"` beibehalten und nur tatsächlich vorhandene Anzeigenamen aus der `dropdown_name`-Zeile der Preset-Tabellen ändern. `Custom *` enthält Namen aus der Tabelle `presets`, keine frei erfundenen `var=wert`-Ausdrücke; die Validierung ersetzt keine fachliche Prüfung.

In [ ]:
if RUN_INSPECTION:
    definitions = inspect_batch(
        workbook, scenarios=scenario_source,
        scenario_filter={"Base Scenario": DEFAULT_BASE},
    )
    preset_columns = [c for c in definitions if c.startswith("Presets_")]
    custom_columns = [c for c in definitions if c.startswith("Custom ")]
    display(definitions[["Scenario Name", "Base Scenario", *preset_columns, *custom_columns]].head(10))
    for column in ("Presets_geometry_table", "Presets_hulls_table", "Presets_heating_system_table"):
        if column in definitions:
            display(definitions[column].value_counts(dropna=False).rename(column))

In [ ]:
TYPOLOGY_PREFIX = "EFH_1G|"  # z. B. Einfamilienhaus; an vorhandene SCEN-Namen anpassen
PILOT_SIZE = 3  # erst kleine Auswahl prüfen; None für alle ausgewählten Typologien

if RUN_INSPECTION:
    typologies = definitions.loc[
        definitions["Scenario Name"].str.startswith(TYPOLOGY_PREFIX, na=False)
    ].copy()
    if typologies.empty:
        raise ValueError("Keine passenden SCEN-Zeilen: Namen oben prüfen und TYPOLOGY_PREFIX anpassen.")
    selected_typologies = typologies if PILOT_SIZE is None else typologies.head(PILOT_SIZE).copy()
    selected_typologies["Base Scenario"] = DEFAULT_BASE
    # Erneut validieren, insbesondere nach manuellen Preset-/Custom-Änderungen.
    scenario_source = inspect_batch(workbook, scenarios=selected_typologies)
    display(scenario_source)
    print(f"Ausgewählt: {len(scenario_source)} von {len(typologies)} Typologiezeilen")

Für den Pilotlauf jetzt `RUN_EXCEL = True` setzen und den folgenden Batch-Abschnitt ausführen. Für den großen Lauf anschließend `PILOT_SIZE = None` setzen, die Auswahl erneut prüfen und **einen neuen `output_dir`** verwenden. `base_reload="always"` lädt `Default` für jede Variante neu; ausgelassene Presets übernehmen dadurch keine Eingaben der vorherigen Variante.

Versionierung: gespeicherte Modellkopie und Auswahl während eines Laufs unverändert lassen. `batch.json` hält Workbook-SHA256, Schema, Optionen und die vollständigen ausgewählten SCEN-Werte fest. Bei veränderten Presets/Modellständen einen neuen Ergebnisordner verwenden. Eine SCEN-CSV enthält Referenzen auf die Basis/Presets, nicht die zugehörige Arbeitsmappe. Die Dateiversion muss deshalb zusammen mit dem CSV aufbewahrt werden.

`batch_size` steuert den Verlustbereich bei einem harten Absturz: noch nicht bestätigte Chunks können erneut gerechnet werden. Für teure Pilotläufe kann `batch_size=1` sinnvoll sein; größere Chunks sparen Exportaufwand. Die Option für gewöhnliches Resume unverändert lassen.

## 2. Jahresergebnisse berechnen und sichern

`batch_size=25` bedeutet höchstens 25 Szenarien pro Jahresergebnisdatei, nicht 25 gleichzeitige
Excel-Berechnungen. Für ungefähr 5.000 Typologien entstehen etwa 200 Jahresdateien.
Stundenwerte bleiben für diesen Anwendungsfall ausgeschaltet.

`base_reload="always"` lädt die Basis vor jedem Szenario und verhindert übernommene Eingaben.
`"first_only"` ist schneller, setzt aber vollständige, unabhängige Presets voraus: die Basis
wird beim ersten ausgeführten Szenario und bei einem Wechsel der Basis geladen, auch nach Resume.
Unvollständige Presets können sonst Werte aus dem vorherigen Szenario übernehmen.

### Fortschritt in CLI und Notebook

Der optionale Callback erhält `(completed, total, scenario_name)`: einmal vor dem Lauf mit
bereits bestätigten Resume-Ergebnissen und leerem Namen, danach nach jedem erfolgreich erfassten
Szenario. Ein laufender VBA-Aufruf liefert keine Zwischenmeldungen. Zwischenstände können
noch nicht gespeicherte Szenarien enthalten; 100 % wird erst nach dem letzten Export gemeldet.
Auch ohne Callback erscheinen Zeitstempel, Phasen, Szenarionamen und Laufzeiten automatisch in Notebook/Terminal und in `batch.log`. Der Callback ist nur eine zusätzliche Anzeige. „Captured“ kann noch ungesicherte Ergebnisse enthalten; „committed“ bedeutet erfolgreich exportiert und im Checkpoint bestätigt.

In [ ]:
def report_progress(completed, total, scenario_name):
    label = scenario_name or "Start / Resume"
    print(f"{completed}/{total} ({completed / total:.0%}) – {label}", flush=True)

In [ ]:
if RUN_EXCEL:
    result = run_batch(
        workbook=workbook,
        scenarios=scenario_source,
        scenario_filter={"Base Scenario": DEFAULT_BASE},
        batch_size=25,
        base_reload="always",
        export_annual=True,
        export_timeseries=False,
        output_dir=output_dir,
        resume=False,
        progress_callback=None,  # automatische Ausgabe funktioniert ohne Callback
    )
    print("Status:", result.status)
    print("Abgeschlossen:", len(result.completed), "bereits vorhanden:", len(result.skipped))
    print("Fehlgeschlagen:", result.failed)
    print("Dateien:", result.exported_files)

## 3. Jahresdaten als Project konsolidieren

Die Jahresdateien enthalten IN, OUT und META im bestehenden Projektformat. Es gibt keinen
separaten Batch-Schema-Mapper. Für die Weiterverarbeitung im Gebäudesektormodell können die
Dateien mit `Project.from_files` zusammengeführt werden. Nur bestätigte Dateien aus dem
Ergebnis verwenden, nicht pauschal alle Dateien eines nach Abbruch verbliebenen Ordners.

Für sehr große Aufträge lassen sich dieselben Dateien einzeln auswerten, um Speicher zu sparen.

In [ ]:
annual_files = []
project = None
if result is not None:
    annual_files = [p for p in result.exported_files if p.suffix == ".xlsx"]
    if annual_files:
        project = Project.from_files(annual_files, unknown="raise")
        assert project.column_names() == result.completed
        summary = pd.DataFrame([
            {"scenario": s.column_name, "QH": s.to_var_dict().get("QH")}
            for s in project
        ])
        display(summary.head())

## 4. Unterbrechen und fortsetzen

Ein explizit von VBA zurückgemeldeter Szenariofehler wird protokolliert und übersprungen, sofern Excel bereit ist. Vor dem nächsten Szenario wird die Basis neu geladen. Nicht wiederherstellbare COM-Probleme stoppen den Lauf; Struktur-/Schema- und Schreibfehler werden nach bestmöglicher Sicherung weitergegeben. `result.failed` enthält Fehler der nicht erfolgreich gespeicherten Szenarien.

Die ausführbaren Resume- und Retry-Beispiele folgen unten. Fremde Excel-Instanzen werden nicht beendet und die Arbeitsmappe wird nicht gespeichert. Ein eigener, eindeutig beendeter Excel-Lauf wird ohne Speichern geschlossen. Ein unsicherer oder blockierter Makrostatus erfordert manuelle Prüfung der protokollierten PID. Der synchrone VBA-Aufruf hat keinen erzwungenen Timeout.

## 5. Optionale Stundenwerte

Für gezielte Kontrollszenarien `export_timeseries=True` und einen separaten Ausgabeordner setzen.
Die API erfasst SIM über `ScenarioTimeseries` und schreibt pro Szenario eine CSV. Die Zuordnung
steht im Manifest `batch.json`; numerische Dateinamen vermeiden Konflikte durch Sonderzeichen.
Die Rohdaten behalten die SIM-Spaltennamen. Einheiten stehen im Zeitreihenschema.

```python
# Nach einem Lauf mit export_timeseries=True:
series_files = [p for p in result.exported_files if p.suffix == ".csv"]
series = pd.read_csv(series_files[0])
print(series.shape, series.columns.tolist())
```

Jahresergebnisse werden mit `read_project` bzw. `Project.from_files` gelesen. Die Stunden-CSV
wird separat gelesen und kann bei Bedarf mit `scenario.set_timeseries(series)` angehängt werden.
Eine Zahl von 8.760 Stunden ist keine universelle Validierung für jedes Kalenderjahr.

## Weiterarbeiten

Das [Python-Angebot](https://simonschaluppe.github.io/peexcel/python.html) zeigt Analysen und
Visualisierungen mit bestehenden Project-Views. Der
[Parameter- und Kennzahlenkatalog](https://simonschaluppe.github.io/peexcel/glossary.html)
erläutert die verfügbaren Variablen.

[Dieses Tutorial als Notebook herunterladen](downloads/tutorial_batch.ipynb).

### Optional: Fortschrittsbalken mit tqdm

`tqdm.auto` wählt eine passende Anzeige für Terminal oder Notebook. Bei Bedarf `pip install tqdm`
installieren. Mit einem Kontextmanager wird der Balken auch bei Fehlern oder Abbruch geschlossen:

```python
from tqdm.auto import tqdm

with tqdm(total=0, desc="PEExcel", unit="Szenario") as bar:
    def show_progress(completed, total, scenario_name):
        bar.total = total
        bar.update(completed - bar.n)
        bar.set_postfix_str(scenario_name or "Start / Resume", refresh=True)

    result = run_batch(
        workbook=workbook,
        scenarios=scenario_source,
        scenario_filter={"Base Scenario": "Default"},
        output_dir=output_dir,
        resume=True,
        progress_callback=show_progress,
    )
```

Die Anzeige zählt erfolgreich erfasste Szenarien. Fehler bleiben in `result.failed` sichtbar.
Ein Fehler im eigenen Callback wird nach Sicherung erfasster Ergebnisse weitergegeben.

## Wiederaufnahme und portable Wiederholung

**Gewöhnliches Resume:** Bei einem bereits bestehenden Batch `RUN_EXCEL = False` und `RUN_RESUME = True` setzen, damit kein neuer Erstlauf im selben Ordner gestartet wird. dieselbe Arbeitsmappe, dieselbe Auswahl und dieselben Exportoptionen verwenden. Bereits bestätigte Ergebnisse werden auch dann ausgelassen, wenn dazwischen Szenarien fehlgeschlagen sind. Jahres- und optionale Zeitreihendateien müssen unverändert vorhanden sein. `batch.json` ist maßgeblich, nicht die Excel-Statusspalte.

Ein `KeyboardInterrupt` sichert vollständig erfasste Ergebnisse und aktualisiert Checkpoint/CSV, soweit Python weiterlaufen kann. Ein synchron blockierter VBA-Aufruf kann den Interrupt verzögern. Nach hartem Abbruch zuerst prüfen, ob Excel noch rechnet; erst danach einen verwaisten `.batch.lock` manuell entfernen. Niemals einen Lock eines laufenden Batches entfernen.

In [ ]:
if RUN_RESUME:
    result = run_batch(
        workbook, scenarios=scenario_source,
        scenario_filter={"Base Scenario": DEFAULT_BASE},
        batch_size=25, base_reload="always", export_annual=True,
        export_timeseries=False, output_dir=output_dir, resume=True,
    )

**Portable Wiederholung:** `RUN_EXCEL = False`, `RUN_RESUME = False` und `RUN_RETRY = True` setzen. Nach einem harten Absturz zunächst `inspect_batch(..., output_dir=...)` ausführen. Es prüft die gespeicherten Dateien und erstellt `remaining_scenarios.csv` neu, ohne Excel oder eine Simulation zu starten. JSON und CSV können nicht gemeinsam atomar ersetzt werden: die CSV kann direkt nach einem Absturz noch inzwischen abgeschlossene Zeilen enthalten. Daher vor dem portablen Retry abgleichen.

Die CSV enthält alle Zeilen ohne bestätigte Ergebnisse, einschließlich fehlgeschlagener, unterbrochener und noch nicht gestarteter Varianten. Ein leerer Batch schreibt nur die Kopfzeile. Nicht zusätzlich gewöhnliches Resume und portable Wiederholung für dieselben verbleibenden Zeilen parallel starten.

In [ ]:
if RUN_RETRY:
    remaining = inspect_batch(workbook, output_dir=output_dir)
    display(remaining)
    if not remaining.empty:
        retry_result = run_batch(
            workbook, scenarios=output_dir / "remaining_scenarios.csv",
            output_dir=output_dir.with_name(output_dir.name + "_retry_01"),
            base_reload="always", export_annual=True, export_timeseries=False,
        )
    else:
        print("Alle ausgewählten Szenarien sind bereits erfolgreich gespeichert.")

## Betrieb auf dem Windows-Simulationsrechner

Python/Notebook-Kernel und Excel laufen lokal auf dem Simulationsrechner in derselben interaktiven Windows-Benutzersitzung. RDP dient zur Bedienung; ein Remote-API-Dienst ist nicht erforderlich. Vor dem großen Lauf den Pilot testen, einschließlich RDP-Verbindung trennen und wiederherstellen. Abmelden, Energiesparen und Neustart können den Lauf beenden; die konkreten Sitzungsrichtlinien prüfen.

`batch.log` zeigt den zuletzt begonnenen Vorgang auch während eines blockierenden Excel-Aufrufs. Unbekannter Makrostatus führt nicht zu einer automatischen Wiederholung. Nur nach nachgewiesenem Ende einer eigenen Excel-Instanz wird einmal neu verbunden. Fremde Instanzen werden nicht beendet; bei unklarem Zustand die protokollierte PID prüfen.

Die Anzeige von `PEI_balance` übernimmt den bestehenden OUT-Wert samt Schemaeinheit. Fehlende/nichtnumerische/unendliche Werte werden gewarnt. Negative Werte sind nicht automatisch Fehler, und ein Saldo muss nicht Null sein. Die Kennzahl ist eine Plausibilitätshilfe und bleibt in den normalen OUT-Exporten erhalten.